# Progress 5 — IdSarcasm Optimization + Modern LLM Experiments

Notebook ini disiapkan untuk Progress 5:

1. **Wajib:** threshold tuning XLM-R Large.
2. **Wajib:** small hyperparameter experiment XLM-R Base → XLM-R Large.
3. **Wajib:** error analysis dari baseline vs optimized.
4. **Tambahan:** Gemma 3n E4B + Qwen3.5-4B + Cendol/Bahasa-4B zero-shot/few-shot.

Catatan: transformer optimization dijalankan di Colab GPU. LM Studio/GGUF dijalankan di PC lokal/WSL, bukan Colab, kecuali memakai tunnel.

## 0. Setup Colab

Jalankan cell ini kalau repo belum ada di runtime Colab. Kalau sudah clone, pakai cell berikutnya untuk `git pull`.

In [ ]:
!git clone https://github.com/feb027/idsarcasm-reproduction.git
%cd idsarcasm-reproduction
!pip install -r requirements.txt

## 0b. Setup jika repo sudah ada

In [ ]:
%cd /content/idsarcasm-reproduction
!git pull
!pip install -r requirements.txt
!mkdir -p results/logs results/optimization results/modern_llm

## 1. Cek status aset Progress 5

In [ ]:
!python --version
!git log --oneline -5
!ls scripts/run_transformer_optimization.py scripts/run_modern_llm_experiments.py docs/progress-5-run-guide.md

## 2. Smoke test transformer optimization

Jalankan ini dulu. Kalau gagal, jangan lanjut full run.

In [ ]:
!python scripts/run_transformer_optimization.py   --dataset twitter   --model xlmr-base   --run-name smoke-twitter-xlmr-base-progress5   --epochs 1   --batch-size 4   --eval-batch-size 8   --max-train-samples 24   --max-eval-samples 12   --max-predict-samples 12   --disable-tqdm

## 3. Wajib 1 — XLM-R Large threshold tuning Twitter

Ini run utama pertama. Simpan log dengan `tee`.

In [ ]:
!python scripts/run_transformer_optimization.py   --dataset twitter   --model xlmr-large   --run-name twitter-xlmr-large-threshold   --epochs 100   --batch-size 32   --eval-batch-size 64   --learning-rate 1e-5   --lr-scheduler-type cosine   --weight-decay 0.03   --label-smoothing-factor 0.0   --max-length 128   --early-stopping-threshold 0.01   --seed 42   --pad-to-max-length   --shuffle-train-dataset   --fp16   --disable-tqdm   2>&1 | tee results/logs/progress-5-optimization-twitter-xlmr-large-threshold.log

## 4. Cek hasil threshold tuning

In [ ]:
import csv, json
from pathlib import Path
p = Path('results/tables/optimization_runs.csv')
if p.exists():
    rows = list(csv.DictReader(p.open(encoding='utf-8')))
    for r in rows:
        print(r['run_id'], 'default_f1=', r['test_default_f1'], 'tuned_f1=', r['test_tuned_f1'], 'delta=', r['delta_f1'], 'thr=', r['selected_threshold'])
else:
    print('Belum ada optimization_runs.csv')

## 5. Wajib 2 — XLM-R Base hyperparameter screening

Jalankan satu per satu. Kalau waktu Colab terbatas, minimal jalankan 3 cell pertama.

In [ ]:
# 5.1 LR rendah
!python scripts/run_transformer_optimization.py --dataset twitter --model xlmr-base --run-name twitter-xlmr-base-lr5e-6-len128 --epochs 100 --batch-size 32 --eval-batch-size 64 --learning-rate 5e-6 --lr-scheduler-type cosine --weight-decay 0.03 --label-smoothing-factor 0.0 --max-length 128 --early-stopping-threshold 0.01 --seed 42 --pad-to-max-length --shuffle-train-dataset --fp16 --disable-tqdm 2>&1 | tee results/logs/progress-5-optimization-twitter-xlmr-base-lr5e-6-len128.log

In [ ]:
# 5.2 LR tinggi
!python scripts/run_transformer_optimization.py --dataset twitter --model xlmr-base --run-name twitter-xlmr-base-lr2e-5-len128 --epochs 100 --batch-size 32 --eval-batch-size 64 --learning-rate 2e-5 --lr-scheduler-type cosine --weight-decay 0.03 --label-smoothing-factor 0.0 --max-length 128 --early-stopping-threshold 0.01 --seed 42 --pad-to-max-length --shuffle-train-dataset --fp16 --disable-tqdm 2>&1 | tee results/logs/progress-5-optimization-twitter-xlmr-base-lr2e-5-len128.log

In [ ]:
# 5.3 Max length 256
!python scripts/run_transformer_optimization.py --dataset twitter --model xlmr-base --run-name twitter-xlmr-base-lr1e-5-len256 --epochs 100 --batch-size 32 --eval-batch-size 64 --learning-rate 1e-5 --lr-scheduler-type cosine --weight-decay 0.03 --label-smoothing-factor 0.0 --max-length 256 --early-stopping-threshold 0.01 --seed 42 --pad-to-max-length --shuffle-train-dataset --fp16 --disable-tqdm 2>&1 | tee results/logs/progress-5-optimization-twitter-xlmr-base-lr1e-5-len256.log

In [ ]:
# 5.4 Weight decay 0.01
!python scripts/run_transformer_optimization.py --dataset twitter --model xlmr-base --run-name twitter-xlmr-base-lr1e-5-wd001 --epochs 100 --batch-size 32 --eval-batch-size 64 --learning-rate 1e-5 --lr-scheduler-type cosine --weight-decay 0.01 --label-smoothing-factor 0.0 --max-length 128 --early-stopping-threshold 0.01 --seed 42 --pad-to-max-length --shuffle-train-dataset --fp16 --disable-tqdm 2>&1 | tee results/logs/progress-5-optimization-twitter-xlmr-base-lr1e-5-wd001.log

In [ ]:
# 5.5 Label smoothing 0.05
!python scripts/run_transformer_optimization.py --dataset twitter --model xlmr-base --run-name twitter-xlmr-base-label-smoothing005 --epochs 100 --batch-size 32 --eval-batch-size 64 --learning-rate 1e-5 --lr-scheduler-type cosine --weight-decay 0.03 --label-smoothing-factor 0.05 --max-length 128 --early-stopping-threshold 0.01 --seed 42 --pad-to-max-length --shuffle-train-dataset --fp16 --disable-tqdm 2>&1 | tee results/logs/progress-5-optimization-twitter-xlmr-base-label-smoothing005.log

## 6. Optional — Reddit XLM-R Large threshold tuning

Jalankan setelah Twitter selesai dan runtime masih cukup.

In [ ]:
!python scripts/run_transformer_optimization.py --dataset reddit --model xlmr-large --run-name reddit-xlmr-large-threshold --epochs 100 --batch-size 32 --eval-batch-size 64 --learning-rate 1e-5 --lr-scheduler-type cosine --weight-decay 0.03 --label-smoothing-factor 0.0 --max-length 128 --early-stopping-threshold 0.01 --seed 42 --pad-to-max-length --shuffle-train-dataset --fp16 --disable-tqdm 2>&1 | tee results/logs/progress-5-optimization-reddit-xlmr-large-threshold.log

## 7. Error analysis quick preview

Cell ini hanya preview. Analisis final dikerjakan setelah hasil dikomit/push dan seluruh output tersedia.

In [ ]:
import csv
from pathlib import Path
pred_path = Path('results/optimization/twitter-xlmr-large-threshold/predictions.csv')
if pred_path.exists():
    rows = [r for r in csv.DictReader(pred_path.open(encoding='utf-8')) if r['split'] == 'test']
    improved = [r for r in rows if r['default_correct'] == '0' and r['tuned_correct'] == '1'][:10]
    worsened = [r for r in rows if r['default_correct'] == '1' and r['tuned_correct'] == '0'][:10]
    print('Improved examples:', len(improved))
    for r in improved[:3]:
        print('
TRUE', r['true_label'], 'default', r['pred_default'], 'tuned', r['pred_tuned'], 'p=', r['prob_sarcastic'])
        print(r['text'][:300])
    print('
Worsened examples:', len(worsened))
    for r in worsened[:3]:
        print('
TRUE', r['true_label'], 'default', r['pred_default'], 'tuned', r['pred_tuned'], 'p=', r['prob_sarcastic'])
        print(r['text'][:300])
else:
    print('predictions.csv belum ada')

## 8. Commit gate setelah transformer run

Setelah threshold + screening selesai, jalankan ini di local/Colab terminal dengan GitHub auth aktif, atau download lalu commit dari PC lokal.

In [ ]:
!git status --short
# Setelah dicek:
# !git add results/tables/optimization_runs.csv results/optimization results/logs/progress-5-optimization-*.log
# !git commit -m "results: add Progress 5 transformer optimization runs"
# !git push

## 9. LM Studio / GGUF — jalankan lokal, bukan Colab

Cell di bawah adalah contoh command untuk terminal lokal/WSL saat LM Studio Local Server aktif di `http://localhost:1234/v1`. Jangan jalankan di Colab kecuali kamu membuat tunnel dari PC lokal.

In [ ]:
# Contoh smoke test lokal:
# python scripts/run_modern_llm_experiments.py --dataset twitter --model local-model --model-alias lmstudio-smoke --api-base http://localhost:1234/v1 --max-samples 5 --print-every 1

# Contoh Gemma zero-shot:
# python scripts/run_modern_llm_experiments.py --dataset twitter --model local-model --model-alias gemma-3n-e4b-gguf --api-base http://localhost:1234/v1 --temperature 0.0 --max-tokens 8 --print-every 50

# Contoh Gemma few-shot:
# python scripts/run_modern_llm_experiments.py --dataset twitter --model local-model --model-alias gemma-3n-e4b-gguf --api-base http://localhost:1234/v1 --few-shot --shots-per-class 2 --temperature 0.0 --max-tokens 8 --print-every 50

# Contoh Qwen few-shot:
# python scripts/run_modern_llm_experiments.py --dataset twitter --model local-model --model-alias qwen3.5-4b-gguf --api-base http://localhost:1234/v1 --few-shot --shots-per-class 2 --temperature 0.0 --max-tokens 8 --print-every 50

# Contoh Cendol/Bahasa-4B few-shot:
# python scripts/run_modern_llm_experiments.py --dataset twitter --model local-model --model-alias cendol-or-bahasa-4b-gguf --api-base http://localhost:1234/v1 --few-shot --shots-per-class 2 --temperature 0.0 --max-tokens 8 --print-every 50

## 10. Setelah run LM Studio selesai

Commit hasil modern LLM:

```bash
git add results/tables/modern_llm_experiments.csv results/modern_llm
git commit -m "results: add Progress 5 modern local LLM experiments"
git push
```

Setelah run selesai, lanjutkan ke pembuatan figure, error analysis final, pembaruan laporan, dan commit akhir.